In [1]:
import pandas as pd
from pathlib import Path

ARTIFACTS_DIR = Path("../artifacts")

labeled_table = pd.read_csv(
    ARTIFACTS_DIR / "labeled_table.csv"
)

print("Shape:", labeled_table.shape)
display(labeled_table.head())

Shape: (96476, 22)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,...,item_count,total_item_price,total_freight_value,unique_products,unique_sellers,total_payment_value,payment_records,max_installments,payment_types,label
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,...,1.0,29.99,8.72,1.0,1.0,38.71,3.0,1.0,"credit_card, voucher",On-time
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,...,1.0,118.70,22.76,1.0,1.0,141.46,1.0,1.0,boleto,On-time
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,...,1.0,159.90,19.22,1.0,1.0,179.12,1.0,3.0,credit_card,On-time
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,...,1.0,45.00,27.20,1.0,1.0,72.20,1.0,1.0,credit_card,On-time
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,...,1.0,19.90,8.72,1.0,1.0,28.62,1.0,1.0,credit_card,On-time


In [2]:
labeled_table["order_purchase_timestamp"] = pd.to_datetime(
    labeled_table["order_purchase_timestamp"],
    errors="coerce"
)

print(
    "Min purchase date:",
    labeled_table["order_purchase_timestamp"].min()
)

print(
    "Max purchase date:",
    labeled_table["order_purchase_timestamp"].max()
)

Min purchase date: 2016-09-15 12:16:38
Max purchase date: 2018-08-29 15:00:37


In [3]:
labeled_table = labeled_table.sort_values(
    "order_purchase_timestamp"
).reset_index(drop=True)

print(labeled_table[
    ["order_purchase_timestamp", "label"]
].head())

print(labeled_table[
    ["order_purchase_timestamp", "label"]
].tail())

  order_purchase_timestamp    label
0      2016-09-15 12:16:38     Late
1      2016-10-03 09:44:50  On-time
2      2016-10-03 16:56:50  On-time
3      2016-10-03 21:01:41  On-time
4      2016-10-03 21:13:36  On-time
      order_purchase_timestamp    label
96471      2018-08-29 12:25:59  On-time
96472      2018-08-29 14:18:23  On-time
96473      2018-08-29 14:18:28  On-time
96474      2018-08-29 14:52:00  On-time
96475      2018-08-29 15:00:37  On-time


In [4]:
n = len(labeled_table)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = labeled_table.iloc[:train_end].copy()
validation = labeled_table.iloc[train_end:val_end].copy()
test = labeled_table.iloc[val_end:].copy()

print("Train shape:", train.shape)
print("Validation shape:", validation.shape)
print("Test shape:", test.shape)

Train shape: (67533, 22)
Validation shape: (14471, 22)
Test shape: (14472, 22)


In [5]:
for name, df in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test)
]:
    print(
        name,
        "| from:",
        df["order_purchase_timestamp"].min(),
        "| to:",
        df["order_purchase_timestamp"].max()
    )

Train | from: 2016-09-15 12:16:38 | to: 2018-04-15 20:07:56
Validation | from: 2018-04-15 20:10:23 | to: 2018-06-21 07:50:39
Test | from: 2018-06-21 08:29:29 | to: 2018-08-29 15:00:37


In [6]:
for name, df in [
    ("Train", train),
    ("Validation", validation),
    ("Test", test)
]:
    print("\n", name)
    print(df["label"].value_counts(normalize=True) * 100)


 Train
label
On-time    90.971821
Late        9.028179
Name: proportion, dtype: float64

 Validation
label
On-time    94.658282
Late        5.341718
Name: proportion, dtype: float64

 Test
label
On-time    93.387231
Late        6.612769
Name: proportion, dtype: float64


In [7]:
train_path = ARTIFACTS_DIR / "train.csv"
validation_path = ARTIFACTS_DIR / "validation.csv"
test_path = ARTIFACTS_DIR / "test.csv"

train.to_csv(train_path, index=False)
validation.to_csv(validation_path, index=False)
test.to_csv(test_path, index=False)

print("Saved:")
print(train_path)
print(validation_path)
print(test_path)

Saved:
..\artifacts\train.csv
..\artifacts\validation.csv
..\artifacts\test.csv


In [8]:
print("""
Notebook 03 Summary
-------------------
- Used a time-based split instead of a random split.
- Reason: the prediction problem is temporal, so training on older orders
  and testing on newer orders better reflects real-world deployment.
- Train: 70%
- Validation: 15%
- Test: 15%
- Label ratios were not forced to be equal because the split is time-based.
""")


Notebook 03 Summary
-------------------
- Used a time-based split instead of a random split.
- Reason: the prediction problem is temporal, so training on older orders
  and testing on newer orders better reflects real-world deployment.
- Train: 70%
- Validation: 15%
- Test: 15%
- Label ratios were not forced to be equal because the split is time-based.

